Getting 15 - minute discharge data from 2021-2025   **netcdf, zarr

In [ ]:
import requests
import pandas as pd
from tqdm.auto import tqdm
import os
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [39]:
# downloading raw discharge data at 15 minute interval from USGS

# ---------------------------------------------------------------
# LOAD GAGES
# ---------------------------------------------------------------

gages_df = pd.read_csv("huc8_gages/huc8_events_and_gages.csv", dtype={"STAID": str})

gage_ids = (
    gages_df["STAID"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.replace(".0", "", regex=False)
    .str.zfill(8)
    .drop_duplicates()
    .tolist()
)

print(f"Found {len(gage_ids)} unique gages")
print(gage_ids[:10])

# ---------------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------------

start_date = "2021-01-01"
end_date   = "2025-12-31"
parameter_code = "00060"

# ---------------------------------------------------------------
# FUNCTION
# ---------------------------------------------------------------

def get_discharge(site):
    url = "https://waterservices.usgs.gov/nwis/iv/"

    params = {
        "format": "json",
        "sites": site,
        "parameterCd": parameter_code,
        "startDT": start_date,
        "endDT": end_date,
        "siteStatus": "all"
    }

    r = requests.get(url, params=params)
    r.raise_for_status()

    js = r.json()
    series = js["value"].get("timeSeries", [])

    if len(series) == 0:
        return pd.DataFrame()

    rows = []

    for ts in series:
        info = ts["sourceInfo"]

        site_no = info["siteCode"][0]["value"]
        site_name = info["siteName"]

        lat = info["geoLocation"]["geogLocation"]["latitude"]
        lon = info["geoLocation"]["geogLocation"]["longitude"]

        units = ts["variable"]["unit"]["unitCode"]

        values = ts["values"][0].get("value", [])

        for obs in values:
            val = obs.get("value")

            rows.append({
                "STAID": site_no,
                "site_name": site_name,
                "datetime": obs["dateTime"],
                "discharge_cfs": float(val) if val not in [None, ""] else None,
                "qualifiers": ",".join(obs.get("qualifiers", [])),
                "units": units,
                "latitude": lat,
                "longitude": lon
            })

    return pd.DataFrame(rows)

# ---------------------------------------------------------------
# DOWNLOAD
# ---------------------------------------------------------------

all_data = []
failed = []
empty = []

for gage in tqdm(gage_ids):
    try:
        df = get_discharge(gage)

        if df.empty:
            empty.append(gage)
        else:
            all_data.append(df)

    except Exception as e:
        failed.append((gage, str(e)))
        print(f"{gage}: {e}")

# ---------------------------------------------------------------
# COMBINE SAFELY
# ---------------------------------------------------------------

print(f"Successful gages: {len(all_data)}")
print(f"Empty gages: {len(empty)}")
print(f"Failed gages: {len(failed)}")

if len(all_data) > 0:
    discharge = pd.concat(all_data, ignore_index=True)

    discharge["datetime"] = pd.to_datetime(discharge["datetime"])
    discharge = discharge.sort_values(["STAID", "datetime"])

    discharge.to_csv("huc8_gages/usgs_discharge_2021_2025.csv", index=False)

    print(discharge.head())

else:
    discharge = pd.DataFrame()
    print("No discharge data found.")
    print("First 20 empty gages:")
    print(empty[:20])
    print("First 20 failed gages:")
    print(failed[:20])

Found 26 unique gages
['02088500', '208726005', '208758850', '02087275', '02087359', '02087324', '208675010', '208735012', '02085070', '02088470']


  0%|          | 0/26 [00:00<?, ?it/s]

Successful gages: 13
Empty gages: 13
Failed gages: 0


/tmp/ipykernel_408/2640948050.py:121: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  discharge["datetime"] = pd.to_datetime(discharge["datetime"])


            STAID                      site_name                   datetime  \
3022955  02085000  ENO RIVER AT HILLSBOROUGH, NC  2021-01-01 00:00:00-05:00   
3022956  02085000  ENO RIVER AT HILLSBOROUGH, NC  2021-01-01 00:15:00-05:00   
3022957  02085000  ENO RIVER AT HILLSBOROUGH, NC  2021-01-01 00:30:00-05:00   
3022958  02085000  ENO RIVER AT HILLSBOROUGH, NC  2021-01-01 00:45:00-05:00   
3022959  02085000  ENO RIVER AT HILLSBOROUGH, NC  2021-01-01 01:00:00-05:00   

         discharge_cfs qualifiers  units   latitude  longitude  
3022955          131.0          A  ft3/s  36.071111 -79.095556  
3022956          131.0          A  ft3/s  36.071111 -79.095556  
3022957          133.0          A  ft3/s  36.071111 -79.095556  
3022958          131.0          A  ft3/s  36.071111 -79.095556  
3022959          131.0          A  ft3/s  36.071111 -79.095556  


In [40]:
print(f"{len(empty)} gages returned no discharge data.")
print(empty[:20])   # first 20

13 gages returned no discharge data.
['208726005', '208758850', '208675010', '208735012', '02088470', '208732885', '208521324', '02085220', '02086000', '208524090', '208524975', '208650112', '208773375']


In [8]:
# changing LST time to UTC

discharge = pd.read_csv(
    "huc8_gages/usgs_discharge_2021_2025.csv",
    dtype={"STAID": str}
)

# Parse mixed-offset Eastern timestamps and convert to UTC
discharge["datetime"] = pd.to_datetime(
    discharge["datetime"],
    errors="coerce",
    utc=True
)

print(discharge["datetime"].isna().sum(), "bad datetime values")

discharge = discharge.dropna(subset=["datetime"])

# Remove timezone while keeping UTC clock time
discharge["datetime"] = discharge["datetime"].dt.tz_localize(None)

# ---------------------------------------------------------------
# Fixed study period in UTC: 2021-2025
# ---------------------------------------------------------------

study_start = pd.Timestamp("2021-01-01 00:00")
study_end   = pd.Timestamp("2025-12-31 23:45")

discharge = discharge[
    (discharge["datetime"] >= study_start) &
    (discharge["datetime"] <= study_end)
].copy()

expected_records = len(pd.date_range(study_start, study_end, freq="15min"))

coverage = []

for gage, df in discharge.groupby("STAID"):

    df = df.sort_values("datetime").set_index("datetime")

    observed_records = (
        df["discharge_cfs"]
        .resample("15min")
        .count()
        .gt(0)
        .sum()
    )

    coverage.append({
        "STAID": gage,
        "observed_records": observed_records,
        "expected_records": expected_records,
        "coverage": observed_records / expected_records,
        "coverage_percent": round(100 * observed_records / expected_records, 2)
    })

coverage = pd.DataFrame(coverage).sort_values("coverage_percent", ascending=False)

coverage.head()


0 bad datetime values


,STAID,observed_records,expected_records,coverage,coverage_percent
10,02088000,175193,175296,0.999412,99.94
6,02087275,175122,175296,0.999007,99.90
8,02087359,174960,175296,0.998083,99.81
7,02087324,174858,175296,0.997501,99.75
11,02088383,172919,175296,0.986440,98.64


In [12]:
discharge.to_csv(
    "huc8_gages/usgs_discharge_2021_2025_UTC.csv",
    index=False
)

print("UTC discharge data saved to huc8_gages/usgs_discharge_2021_2025_UTC.csv")

UTC discharge data saved to huc8_gages/usgs_discharge_2021_2025_UTC.csv


In [13]:
raw = pd.read_csv("huc8_gages/usgs_discharge_2021_2025_UTC.csv", nrows=5)

print(raw["datetime"])

0    2021-01-01 05:00:00
1    2021-01-01 05:15:00
2    2021-01-01 05:30:00
3    2021-01-01 05:45:00
4    2021-01-01 06:00:00
Name: datetime, dtype: object


In [14]:
raw = pd.read_csv("huc8_gages/usgs_discharge_2021_2025_UTC.csv")

print(raw["datetime"].str[-6:].value_counts())

datetime
:15:00    536735
:45:00    536511
:30:00    535780
:00:00    535445
:25:00    188994
:50:00    188991
:55:00    188985
:05:00    188963
:35:00    188902
:40:00    188871
:10:00    188864
:20:00    188858
:23:00       257
:07:00        30
:09:00        17
:46:00        10
:22:00        10
:34:00         6
:58:00         5
:01:00         4
:14:00         3
:51:00         3
:53:00         3
:52:00         3
:54:00         3
:37:00         2
:28:00         2
:04:00         2
:08:00         2
:48:00         2
:26:00         2
:24:00         2
:13:00         2
:11:00         2
:29:00         2
:57:00         1
:31:00         1
:33:00         1
:32:00         1
:02:00         1
:16:00         1
:18:00         1
:19:00         1
:03:00         1
:27:00         1
:47:00         1
:56:00         1
:49:00         1
:06:00         1
:12:00         1
:38:00         1
:41:00         1
Name: count, dtype: int64


Resampling to 15 min intervals for all gages

In [16]:
# resampling to 15 minute intervals

# ---------------------------------------------------------------
# RESAMPLE TO 15-MINUTE UTC GRID
# ---------------------------------------------------------------

discharge_15min = (
    discharge
    .set_index("datetime")
    .groupby("STAID")
    .resample("15min")[
        ["discharge_cfs", "latitude", "longitude"]
    ]
    .max()
    .reset_index()
)

# Add site names back
site_names = (
    discharge[["STAID", "site_name"]]
    .drop_duplicates()
)

discharge_15min = discharge_15min.merge(
    site_names,
    on="STAID",
    how="left"
)

# Reorder columns
discharge_15min = discharge_15min[
    [
        "STAID",
        "site_name",
        "datetime",
        "discharge_cfs",
        "latitude",
        "longitude",
    ]
]

discharge_15min = discharge_15min.sort_values(
    ["STAID", "datetime"]
)

print(discharge_15min.head())

      STAID                      site_name            datetime  discharge_cfs  \
0  02085000  ENO RIVER AT HILLSBOROUGH, NC 2021-01-01 05:00:00          131.0   
1  02085000  ENO RIVER AT HILLSBOROUGH, NC 2021-01-01 05:15:00          131.0   
2  02085000  ENO RIVER AT HILLSBOROUGH, NC 2021-01-01 05:30:00          133.0   
3  02085000  ENO RIVER AT HILLSBOROUGH, NC 2021-01-01 05:45:00          131.0   
4  02085000  ENO RIVER AT HILLSBOROUGH, NC 2021-01-01 06:00:00          131.0   

    latitude  longitude  
0  36.071111 -79.095556  
1  36.071111 -79.095556  
2  36.071111 -79.095556  
3  36.071111 -79.095556  
4  36.071111 -79.095556  


In [22]:
summary = (
    discharge_15min
    .assign(minute=discharge_15min["datetime"].dt.minute)
    .groupby("STAID")["minute"]
    .apply(lambda x: sorted(x.unique()))
)

print(summary)

STAID
02085000    [0, 15, 30, 45]
02085070    [0, 15, 30, 45]
02085500    [0, 15, 30, 45]
02086500    [0, 15, 30, 45]
02086624    [0, 15, 30, 45]
02086849    [0, 15, 30, 45]
02087275    [0, 15, 30, 45]
02087324    [0, 15, 30, 45]
02087359    [0, 15, 30, 45]
02087580    [0, 15, 30, 45]
02088000    [0, 15, 30, 45]
02088383    [0, 15, 30, 45]
02088500    [0, 15, 30, 45]
Name: minute, dtype: object


In [23]:
discharge_15min.to_csv(
    "huc8_gages/usgs_discharge_2021_2025_UTC_15min.csv",
    index=False
)

print("Saved UTC 15-minute discharge data.")

Saved UTC 15-minute discharge data.


In [24]:
df = pd.read_csv("huc8_gages/usgs_discharge_2021_2025_UTC_15min.csv")

print(df.columns)

Index(['STAID', 'site_name', 'datetime', 'discharge_cfs', 'latitude',
       'longitude'],
      dtype='object')


In [42]:
qualifiers = (
    discharge.groupby("STAID")["qualifiers"]
    .unique()
    .reset_index()
)

print(qualifiers.to_string(index=False))

   STAID       qualifiers
02085000         [A, A,e]
02085070 [A, A,e, P, P,e]
02085500         [A, A,e]
02086500         [A, A,e]
02086624         [A, A,e]
02086849         [A,e, A]
02087275         [A, A,e]
02087324    [A, A,e, A,R]
02087359    [A,R, A, A,e]
02087580    [A, A,R, A,e]
02088000      [A, A,e, P]
02088383         [A, A,e]
02088500         [A, A,e]


In [43]:
for gage, df in discharge.groupby("STAID"):
    print(f"\n{gage}")
    print(df["qualifiers"].value_counts())


02085000
qualifiers
A      172174
A,e       216
Name: count, dtype: int64

02085070
qualifiers
A      315762
P       83038
A,e       314
P,e        16
Name: count, dtype: int64

02085500
qualifiers
A      150450
A,e      1697
Name: count, dtype: int64

02086500
qualifiers
A      131702
A,e      2539
Name: count, dtype: int64

02086624
qualifiers
A      148706
A,e      5509
Name: count, dtype: int64

02086849
qualifiers
A      162732
A,e      2144
Name: count, dtype: int64

02087275
qualifiers
A      524077
A,e       968
Name: count, dtype: int64

02087324
qualifiers
A      437182
A,R      2599
A,e        71
Name: count, dtype: int64

02087359
qualifiers
A      488664
A,R     33803
A,e      2134
Name: count, dtype: int64

02087580
qualifiers
A      419981
A,R     56680
A,e       704
Name: count, dtype: int64

02088000
qualifiers
A      173918
P        1294
A,e         2
Name: count, dtype: int64

02088383
qualifiers
A      172776
A,e       163
Name: count, dtype: int64

02088500
qualif

In [27]:
# plotting

# ---------------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------------

discharge = pd.read_csv(
    "huc8_gages/usgs_discharge_2021_2025_UTC_15min.csv",
    dtype={"STAID": str}
)

discharge["datetime"] = pd.to_datetime(
    discharge["datetime"],
    errors="coerce"
)

discharge = discharge.dropna(subset=["datetime"])

# Optional fixed study period
study_start = pd.Timestamp("2021-01-01 00:00")
study_end   = pd.Timestamp("2025-12-31 23:45")

discharge = discharge[
    (discharge["datetime"] >= study_start) &
    (discharge["datetime"] <= study_end)
].copy()

# ---------------------------------------------------------------
# OUTPUT FOLDER
# ---------------------------------------------------------------

out_dir = "huc8_gages/discharge_plots"
os.makedirs(out_dir, exist_ok=True)

# ---------------------------------------------------------------
# PLOT ONE FIGURE PER GAGE
# ---------------------------------------------------------------

for gage, df in discharge.groupby("STAID"):

    df = df.sort_values("datetime")

    site_name = df["site_name"].iloc[0]

    fig, ax = plt.subplots(figsize=(14, 4.5))

    ax.plot(
        df["datetime"],
        df["discharge_cfs"],
        linewidth=0.6
    )

    ax.set_title(
        f"USGS {gage} — {site_name}\nDischarge Time Series, 2021–2025",
        fontsize=13
    )

    ax.set_xlabel("Date (UTC)")
    ax.set_ylabel("Discharge (cfs)")

    ax.grid(True, alpha=0.3)

    # Nice yearly ticks
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.set_xlim(study_start, study_end)

    fig.tight_layout()

    out_path = os.path.join(out_dir, f"{gage}_discharge_2021_2025.png")
    fig.savefig(out_path, dpi=300, bbox_inches="tight")

    plt.close(fig)

print(f"Saved plots to: {out_dir}")

Saved plots to: huc8_gages/discharge_plots


In [28]:
# checking for gaps in the record

gap_summary = []

for gage, df in discharge_15min.groupby("STAID"):

    df = df.sort_values("datetime").copy()

    missing = df["discharge_cfs"].isna()

    groups = (missing != missing.shift()).cumsum()

    gaps = (
        df[missing]
        .groupby(groups[missing])
        .size()
    )

    largest_gap = gaps.max() if len(gaps) else 0

    gap_summary.append({
        "STAID": gage,
        "largest_gap_records": largest_gap,
        "largest_gap_days": round(largest_gap * 15 / 60 / 24, 2)
    })

gap_summary = pd.DataFrame(gap_summary)

print(gap_summary.sort_values("largest_gap_days", ascending=False))

       STAID  largest_gap_records  largest_gap_days
5   02086849                   49              0.51
9   02087580                   49              0.51
4   02086624                   47              0.49
3   02086500                   47              0.49
0   02085000                   30              0.31
2   02085500                   16              0.17
1   02085070                   15              0.16
10  02088000                   15              0.16
8   02087359                   15              0.16
11  02088383                   15              0.16
12  02088500                   15              0.16
7   02087324                    7              0.07
6   02087275                    7              0.07


In [1]:
# removing rows with nonvalid gages

empty = [
    '208726005', '208758850', '208675010', '208735012',
    '02088470', '208732885', '208521324', '02085220',
    '02086000', '208524090', '208524975', '208650112',
    '208773375'
]

# Load CSV, forcing STAID as string
combined = pd.read_csv(
    "huc8_gages/huc8_storms_plus_extra_gages_with_divides_filtered.csv",
    dtype={"STAID": str}
)

# Clean STAID formatting
combined["STAID_clean"] = (
    combined["STAID"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(8)
)

empty_clean = (
    pd.Series(empty)
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(8)
)

# Remove gages with no discharge data
combined = combined[
    ~combined["STAID_clean"].isin(empty_clean)
].copy()

# Drop helper column
combined = combined.drop(columns="STAID_clean")

# Save
combined.to_csv(
    "huc8_gages/huc8_events_and_gages_unique_with_divides_filtered.csv",
    index=False
)

print(f"Saved {len(combined)} rows after removing no-data gages.")

Saved 66 rows after removing no-data gages.
